# 🧠 高级调度和内存管理

## 📚 学习目标
- 🏗️ 理解LLM推理系统中的调度机制
- ⚡ 掌握Continuous Batching的实现原理
- 🔧 学会内存感知的调度策略
- 📊 了解性能监控和优化技巧

## 🎯 核心概念
1. **请求调度**: 如何高效管理多个推理请求
2. **内存管理**: PagedAttention和KV Cache优化
3. **批处理策略**: 动态批处理和连续批处理
4. **性能优化**: 延迟、吞吐量和资源利用率的平衡

## 🔧 环境设置

In [ ]:
# 检测是否在Colab环境中运行
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔄 在Google Colab中运行，正在设置环境...")
    
    # 克隆项目仓库
    !git clone https://github.com/your-username/nano-vllm-learning.git
    %cd nano-vllm-learning
    
    # 安装依赖
    !pip install torch transformers numpy matplotlib seaborn tqdm psutil
else:
    print("💻 在本地环境中运行")

# 导入必要的库
import os
import time
import random
import threading
import heapq
import uuid
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any, Deque
from enum import Enum
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import psutil

# 设置随机种子
random.seed(42)
np.random.seed(42)

print("✅ 环境设置完成！")

## 📋 请求管理系统

首先实现请求的生命周期管理，包括请求状态、优先级和资源需求。

In [ ]:
class RequestStatus(Enum):
    """请求状态枚举"""
    WAITING = "waiting"      # 等待调度
    RUNNING = "running"      # 正在执行
    SWAPPED = "swapped"      # 被换出内存
    FINISHED = "finished"    # 执行完成
    CANCELLED = "cancelled"  # 被取消


class SchedulerPolicy(Enum):
    """调度策略枚举"""
    FCFS = "fcfs"                    # 先来先服务
    SJF = "sjf"                      # 最短作业优先
    PRIORITY = "priority"            # 优先级调度
    MEMORY_AWARE = "memory_aware"    # 内存感知调度


@dataclass
class InferenceRequest:
    """推理请求数据结构"""
    request_id: str
    prompt: str
    max_tokens: int = 100
    temperature: float = 0.7
    priority: int = 1  # 1-10, 数字越大优先级越高
    
    # 时间戳
    created_at: float = field(default_factory=time.time)
    started_at: Optional[float] = None
    finished_at: Optional[float] = None
    
    # 状态管理
    status: RequestStatus = RequestStatus.WAITING
    
    # 资源需求估算
    estimated_tokens: int = 0
    estimated_memory: int = 0  # MB
    
    # 生成结果
    generated_tokens: List[int] = field(default_factory=list)
    generated_text: str = ""
    
    def __post_init__(self):
        """初始化后处理"""
        # 估算资源需求
        prompt_length = len(self.prompt.split())
        self.estimated_tokens = prompt_length + self.max_tokens
        # 简化的内存估算：每个token约4KB
        self.estimated_memory = self.estimated_tokens * 4 // 1024  # MB
    
    @property
    def wait_time(self) -> float:
        """等待时间"""
        if self.started_at:
            return self.started_at - self.created_at
        return time.time() - self.created_at
    
    @property
    def execution_time(self) -> Optional[float]:
        """执行时间"""
        if self.started_at and self.finished_at:
            return self.finished_at - self.started_at
        elif self.started_at:
            return time.time() - self.started_at
        return None


# 创建示例请求
def create_sample_requests(num_requests: int = 10) -> List[InferenceRequest]:
    """创建示例请求"""
    prompts = [
        "解释什么是人工智能",
        "写一首关于春天的诗",
        "Python中的装饰器是什么",
        "如何制作美味的意大利面",
        "机器学习的基本概念",
        "描述一下你最喜欢的电影",
        "什么是区块链技术",
        "如何学习一门新的编程语言",
        "解释量子计算的原理",
        "写一个简短的科幻故事"
    ]
    
    requests = []
    for i in range(num_requests):
        request = InferenceRequest(
            request_id=f"req_{uuid.uuid4().hex[:8]}",
            prompt=random.choice(prompts),
            max_tokens=random.randint(50, 200),
            temperature=random.uniform(0.1, 1.0),
            priority=random.randint(1, 5)
        )
        requests.append(request)
        # 模拟请求间隔
        time.sleep(0.01)
    
    return requests

# 创建示例请求并显示
sample_requests = create_sample_requests(5)
print("📋 示例请求列表:")
for req in sample_requests:
    print(f"  {req.request_id}: '{req.prompt[:30]}...' (优先级: {req.priority}, 预估内存: {req.estimated_memory}MB)")

## 🧠 内存管理器

实现PagedAttention风格的内存管理，支持动态分配和回收。

In [ ]:
@dataclass
class MemoryBlock:
    """内存块"""
    block_id: int
    size: int  # MB
    is_free: bool = True
    request_id: Optional[str] = None
    allocated_at: Optional[float] = None


class MemoryManager:
    """内存管理器 - 模拟PagedAttention的内存管理"""
    
    def __init__(self, total_memory: int = 1024):  # MB
        self.total_memory = total_memory
        self.block_size = 16  # 每个块16MB
        self.num_blocks = total_memory // self.block_size
        
        # 初始化内存块
        self.blocks = [
            MemoryBlock(block_id=i, size=self.block_size)
            for i in range(self.num_blocks)
        ]
        
        self.allocation_history = []
        self._lock = threading.Lock()
    
    def get_memory_stats(self) -> Dict[str, Any]:
        """获取内存统计信息"""
        with self._lock:
            free_blocks = sum(1 for block in self.blocks if block.is_free)
            used_blocks = self.num_blocks - free_blocks
            
            return {
                'total_memory': self.total_memory,
                'used_memory': used_blocks * self.block_size,
                'free_memory': free_blocks * self.block_size,
                'utilization': used_blocks / self.num_blocks,
                'fragmentation': self._calculate_fragmentation()
            }
    
    def _calculate_fragmentation(self) -> float:
        """计算内存碎片率"""
        # 简化的碎片计算：连续空闲块的数量
        free_segments = 0
        in_free_segment = False
        
        for block in self.blocks:
            if block.is_free:
                if not in_free_segment:
                    free_segments += 1
                    in_free_segment = True
            else:
                in_free_segment = False
        
        total_free = sum(1 for block in self.blocks if block.is_free)
        if total_free == 0:
            return 0.0
        
        return free_segments / total_free
    
    def allocate(self, request: InferenceRequest) -> bool:
        """为请求分配内存"""
        required_blocks = (request.estimated_memory + self.block_size - 1) // self.block_size
        
        with self._lock:
            # 寻找连续的空闲块
            allocated_blocks = []
            consecutive_free = 0
            
            for i, block in enumerate(self.blocks):
                if block.is_free:
                    consecutive_free += 1
                    if consecutive_free >= required_blocks:
                        # 找到足够的连续块
                        start_idx = i - required_blocks + 1
                        allocated_blocks = list(range(start_idx, i + 1))
                        break
                else:
                    consecutive_free = 0
            
            if len(allocated_blocks) < required_blocks:
                return False  # 内存不足
            
            # 分配内存块
            for block_idx in allocated_blocks:
                self.blocks[block_idx].is_free = False
                self.blocks[block_idx].request_id = request.request_id
                self.blocks[block_idx].allocated_at = time.time()
            
            self.allocation_history.append({
                'request_id': request.request_id,
                'blocks': allocated_blocks,
                'allocated_at': time.time()
            })
            
            return True
    
    def deallocate(self, request_id: str) -> bool:
        """释放请求的内存"""
        with self._lock:
            deallocated = False
            for block in self.blocks:
                if block.request_id == request_id:
                    block.is_free = True
                    block.request_id = None
                    block.allocated_at = None
                    deallocated = True
            
            return deallocated
    
    def can_allocate(self, request: InferenceRequest) -> bool:
        """检查是否可以分配内存"""
        required_blocks = (request.estimated_memory + self.block_size - 1) // self.block_size
        
        with self._lock:
            consecutive_free = 0
            for block in self.blocks:
                if block.is_free:
                    consecutive_free += 1
                    if consecutive_free >= required_blocks:
                        return True
                else:
                    consecutive_free = 0
            
            return False


# 测试内存管理器
memory_manager = MemoryManager(total_memory=512)  # 512MB

print("🧠 内存管理器测试:")
print(f"初始状态: {memory_manager.get_memory_stats()}")

# 分配一些内存
for i, req in enumerate(sample_requests[:3]):
    success = memory_manager.allocate(req)
    print(f"为请求 {req.request_id} 分配内存: {'成功' if success else '失败'}")

print(f"分配后状态: {memory_manager.get_memory_stats()}")

# 释放内存
memory_manager.deallocate(sample_requests[0].request_id)
print(f"释放后状态: {memory_manager.get_memory_stats()}")

## 🎯 智能调度器

实现多种调度策略，包括FCFS、优先级调度和内存感知调度。

In [ ]:
class IntelligentScheduler:
    """智能调度器"""
    
    def __init__(self, 
                 memory_manager: MemoryManager,
                 policy: SchedulerPolicy = SchedulerPolicy.MEMORY_AWARE,
                 max_batch_size: int = 8):
        self.memory_manager = memory_manager
        self.policy = policy
        self.max_batch_size = max_batch_size
        
        # 请求队列
        self.waiting_queue: Deque[InferenceRequest] = deque()
        self.running_requests: Dict[str, InferenceRequest] = {}
        self.swapped_requests: Dict[str, InferenceRequest] = {}
        self.finished_requests: List[InferenceRequest] = []
        
        # 统计信息
        self.stats = {
            'total_requests': 0,
            'completed_requests': 0,
            'average_wait_time': 0.0,
            'average_execution_time': 0.0,
            'throughput': 0.0
        }
        
        self._lock = threading.Lock()
    
    def add_request(self, request: InferenceRequest):
        """添加新请求"""
        with self._lock:
            self.waiting_queue.append(request)
            self.stats['total_requests'] += 1
    
    def _sort_waiting_queue(self):
        """根据调度策略排序等待队列"""
        if self.policy == SchedulerPolicy.FCFS:
            # 先来先服务，保持原顺序
            pass
        elif self.policy == SchedulerPolicy.SJF:
            # 最短作业优先
            self.waiting_queue = deque(sorted(
                self.waiting_queue, 
                key=lambda x: x.estimated_tokens
            ))
        elif self.policy == SchedulerPolicy.PRIORITY:
            # 优先级调度
            self.waiting_queue = deque(sorted(
                self.waiting_queue, 
                key=lambda x: (-x.priority, x.created_at)
            ))
        elif self.policy == SchedulerPolicy.MEMORY_AWARE:
            # 内存感知调度：优先级 + 内存效率
            def memory_score(req):
                memory_efficiency = 1.0 / (req.estimated_memory + 1)
                priority_score = req.priority / 10.0
                wait_penalty = min(req.wait_time / 60.0, 1.0)  # 等待时间惩罚
                return priority_score + memory_efficiency + wait_penalty
            
            self.waiting_queue = deque(sorted(
                self.waiting_queue, 
                key=memory_score,
                reverse=True
            ))
    
    def schedule_batch(self) -> List[InferenceRequest]:
        """调度一个批次的请求"""
        with self._lock:
            if not self.waiting_queue:
                return []
            
            # 排序等待队列
            self._sort_waiting_queue()
            
            # 选择可以调度的请求
            batch = []
            temp_queue = deque()
            
            while (self.waiting_queue and 
                   len(batch) < self.max_batch_size and
                   len(self.running_requests) + len(batch) < self.max_batch_size):
                
                request = self.waiting_queue.popleft()
                
                # 检查内存是否足够
                if self.memory_manager.can_allocate(request):
                    # 分配内存
                    if self.memory_manager.allocate(request):
                        request.status = RequestStatus.RUNNING
                        request.started_at = time.time()
                        batch.append(request)
                        self.running_requests[request.request_id] = request
                    else:
                        # 分配失败，放回队列
                        temp_queue.append(request)
                else:
                    # 内存不足，尝试换出一些请求
                    if self._try_swap_out():
                        # 换出成功，重新尝试
                        self.waiting_queue.appendleft(request)
                    else:
                        # 无法换出，放回队列
                        temp_queue.append(request)
            
            # 将无法调度的请求放回队列
            while temp_queue:
                self.waiting_queue.appendleft(temp_queue.pop())
            
            return batch
    
    def _try_swap_out(self) -> bool:
        """尝试换出一些运行中的请求以释放内存"""
        # 简化的换出策略：换出优先级最低的请求
        if not self.running_requests:
            return False
        
        # 找到优先级最低的请求
        min_priority_req = min(
            self.running_requests.values(),
            key=lambda x: x.priority
        )
        
        # 换出请求
        self.swap_out_request(min_priority_req.request_id)
        return True
    
    def swap_out_request(self, request_id: str):
        """换出请求"""
        if request_id in self.running_requests:
            request = self.running_requests.pop(request_id)
            request.status = RequestStatus.SWAPPED
            self.swapped_requests[request_id] = request
            self.memory_manager.deallocate(request_id)
    
    def complete_request(self, request_id: str, generated_text: str = ""):
        """完成请求"""
        with self._lock:
            request = None
            
            if request_id in self.running_requests:
                request = self.running_requests.pop(request_id)
            elif request_id in self.swapped_requests:
                request = self.swapped_requests.pop(request_id)
            
            if request:
                request.status = RequestStatus.FINISHED
                request.finished_at = time.time()
                request.generated_text = generated_text
                self.finished_requests.append(request)
                self.memory_manager.deallocate(request_id)
                self.stats['completed_requests'] += 1
                
                # 更新统计信息
                self._update_stats()
    
    def _update_stats(self):
        """更新统计信息"""
        if not self.finished_requests:
            return
        
        wait_times = [req.wait_time for req in self.finished_requests]
        exec_times = [req.execution_time for req in self.finished_requests if req.execution_time]
        
        self.stats['average_wait_time'] = np.mean(wait_times)
        if exec_times:
            self.stats['average_execution_time'] = np.mean(exec_times)
        
        # 计算吞吐量 (请求/秒)
        if len(self.finished_requests) > 1:
            time_span = (self.finished_requests[-1].finished_at - 
                        self.finished_requests[0].started_at)
            if time_span > 0:
                self.stats['throughput'] = len(self.finished_requests) / time_span
    
    def get_status(self) -> Dict[str, Any]:
        """获取调度器状态"""
        with self._lock:
            return {
                'policy': self.policy.value,
                'waiting_requests': len(self.waiting_queue),
                'running_requests': len(self.running_requests),
                'swapped_requests': len(self.swapped_requests),
                'finished_requests': len(self.finished_requests),
                'memory_stats': self.memory_manager.get_memory_stats(),
                'performance_stats': self.stats.copy()
            }


# 测试调度器
scheduler = IntelligentScheduler(
    memory_manager=memory_manager,
    policy=SchedulerPolicy.MEMORY_AWARE,
    max_batch_size=4
)

print("🎯 调度器测试:")

# 添加请求
test_requests = create_sample_requests(8)
for req in test_requests:
    scheduler.add_request(req)

print(f"添加了 {len(test_requests)} 个请求")
print(f"调度器状态: {scheduler.get_status()}")

# 调度第一个批次
batch = scheduler.schedule_batch()
print(f"\n调度了 {len(batch)} 个请求:")
for req in batch:
    print(f"  - {req.request_id}: 优先级={req.priority}, 内存={req.estimated_memory}MB")

print(f"\n调度后状态: {scheduler.get_status()}")

## 🔄 连续批处理 (Continuous Batching)

实现连续批处理机制，允许动态添加和移除请求。

In [ ]:
class ContinuousBatchingEngine:
    """连续批处理引擎"""
    
    def __init__(self, scheduler: IntelligentScheduler):
        self.scheduler = scheduler
        self.is_running = False
        self.processing_thread = None
        
        # 模拟推理参数
        self.tokens_per_second = 50  # 每秒生成的token数
        self.processing_interval = 0.1  # 处理间隔(秒)
        
        # 性能监控
        self.performance_history = []
    
    def start(self):
        """启动连续批处理"""
        if self.is_running:
            return
        
        self.is_running = True
        self.processing_thread = threading.Thread(target=self._processing_loop)
        self.processing_thread.daemon = True
        self.processing_thread.start()
        print("🔄 连续批处理引擎已启动")
    
    def stop(self):
        """停止连续批处理"""
        self.is_running = False
        if self.processing_thread:
            self.processing_thread.join()
        print("⏹️ 连续批处理引擎已停止")
    
    def _processing_loop(self):
        """主处理循环"""
        while self.is_running:
            try:
                # 调度新的批次
                new_batch = self.scheduler.schedule_batch()
                
                # 处理当前运行的请求
                self._process_running_requests()
                
                # 记录性能数据
                self._record_performance()
                
                time.sleep(self.processing_interval)
                
            except Exception as e:
                print(f"处理循环错误: {e}")
                time.sleep(0.1)
    
    def _process_running_requests(self):
        """处理运行中的请求"""
        completed_requests = []
        
        for request_id, request in list(self.scheduler.running_requests.items()):
            # 模拟token生成
            if request.execution_time:
                generated_tokens = int(request.execution_time * self.tokens_per_second)
                
                # 检查是否完成
                if generated_tokens >= request.max_tokens:
                    # 生成模拟文本
                    generated_text = f"生成的回复: {request.prompt[:20]}... (共{generated_tokens}个token)"
                    completed_requests.append((request_id, generated_text))
        
        # 完成请求
        for request_id, generated_text in completed_requests:
            self.scheduler.complete_request(request_id, generated_text)
    
    def _record_performance(self):
        """记录性能数据"""
        status = self.scheduler.get_status()
        memory_stats = status['memory_stats']
        
        performance_data = {
            'timestamp': time.time(),
            'waiting_requests': status['waiting_requests'],
            'running_requests': status['running_requests'],
            'memory_utilization': memory_stats['utilization'],
            'throughput': status['performance_stats']['throughput']
        }
        
        self.performance_history.append(performance_data)
        
        # 保持历史记录在合理范围内
        if len(self.performance_history) > 1000:
            self.performance_history = self.performance_history[-500:]
    
    def add_request(self, request: InferenceRequest):
        """动态添加请求"""
        self.scheduler.add_request(request)
    
    def get_performance_summary(self) -> Dict[str, Any]:
        """获取性能摘要"""
        if not self.performance_history:
            return {}
        
        recent_data = self.performance_history[-100:]  # 最近100个数据点
        
        return {
            'avg_waiting_requests': np.mean([d['waiting_requests'] for d in recent_data]),
            'avg_running_requests': np.mean([d['running_requests'] for d in recent_data]),
            'avg_memory_utilization': np.mean([d['memory_utilization'] for d in recent_data]),
            'max_throughput': max([d['throughput'] for d in recent_data] + [0]),
            'scheduler_status': self.scheduler.get_status()
        }


# 创建连续批处理引擎
engine = ContinuousBatchingEngine(scheduler)

print("🔄 连续批处理演示:")
print("启动引擎...")
engine.start()

# 动态添加请求
print("\n动态添加请求...")
for i in range(5):
    new_request = InferenceRequest(
        request_id=f"dynamic_req_{i}",
        prompt=f"动态请求 {i}: 请解释机器学习的基本概念",
        max_tokens=random.randint(30, 80),
        priority=random.randint(1, 3)
    )
    engine.add_request(new_request)
    print(f"  添加请求: {new_request.request_id}")
    time.sleep(0.5)

# 运行一段时间
print("\n运行处理...")
time.sleep(3)

# 获取性能摘要
performance = engine.get_performance_summary()
print(f"\n性能摘要: {performance}")

# 停止引擎
engine.stop()

## 📊 性能可视化

可视化调度器的性能指标和内存使用情况。

In [ ]:
def visualize_scheduler_performance(engine: ContinuousBatchingEngine):
    """可视化调度器性能"""
    if not engine.performance_history:
        print("没有性能数据可供可视化")
        return
    
    # 准备数据
    timestamps = [d['timestamp'] for d in engine.performance_history]
    waiting_requests = [d['waiting_requests'] for d in engine.performance_history]
    running_requests = [d['running_requests'] for d in engine.performance_history]
    memory_utilization = [d['memory_utilization'] * 100 for d in engine.performance_history]
    throughput = [d['throughput'] for d in engine.performance_history]
    
    # 转换时间戳为相对时间
    start_time = timestamps[0]
    relative_times = [(t - start_time) for t in timestamps]
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('🎯 调度器性能监控', fontsize=16, fontweight='bold')
    
    # 1. 请求队列长度
    axes[0, 0].plot(relative_times, waiting_requests, 'b-', label='等待中', linewidth=2)
    axes[0, 0].plot(relative_times, running_requests, 'g-', label='运行中', linewidth=2)
    axes[0, 0].set_title('📋 请求队列状态')
    axes[0, 0].set_xlabel('时间 (秒)')
    axes[0, 0].set_ylabel('请求数量')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 内存利用率
    axes[0, 1].plot(relative_times, memory_utilization, 'r-', linewidth=2)
    axes[0, 1].fill_between(relative_times, memory_utilization, alpha=0.3, color='red')
    axes[0, 1].set_title('🧠 内存利用率')
    axes[0, 1].set_xlabel('时间 (秒)')
    axes[0, 1].set_ylabel('利用率 (%)')
    axes[0, 1].set_ylim(0, 100)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 吞吐量
    axes[1, 0].plot(relative_times, throughput, 'purple', linewidth=2)
    axes[1, 0].set_title('⚡ 系统吞吐量')
    axes[1, 0].set_xlabel('时间 (秒)')
    axes[1, 0].set_ylabel('请求/秒')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. 请求分布饼图
    status = engine.scheduler.get_status()
    labels = ['等待中', '运行中', '已完成']
    sizes = [
        status['waiting_requests'],
        status['running_requests'],
        status['finished_requests']
    ]
    colors = ['lightblue', 'lightgreen', 'lightcoral']
    
    # 过滤掉为0的数据
    filtered_data = [(label, size, color) for label, size, color in zip(labels, sizes, colors) if size > 0]
    if filtered_data:
        labels, sizes, colors = zip(*filtered_data)
        axes[1, 1].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[1, 1].set_title('📊 请求状态分布')
    
    plt.tight_layout()
    plt.show()


def compare_scheduling_policies():
    """比较不同调度策略的性能"""
    policies = [SchedulerPolicy.FCFS, SchedulerPolicy.PRIORITY, SchedulerPolicy.MEMORY_AWARE]
    results = {}
    
    print("🔍 调度策略性能比较:")
    
    for policy in policies:
        print(f"\n测试策略: {policy.value}")
        
        # 创建新的内存管理器和调度器
        test_memory = MemoryManager(total_memory=256)
        test_scheduler = IntelligentScheduler(
            memory_manager=test_memory,
            policy=policy,
            max_batch_size=4
        )
        
        # 创建测试请求
        test_requests = create_sample_requests(10)
        for req in test_requests:
            test_scheduler.add_request(req)
        
        # 模拟处理
        start_time = time.time()
        processed = 0
        
        while test_scheduler.waiting_queue and processed < 10:
            batch = test_scheduler.schedule_batch()
            
            # 模拟处理时间
            for req in batch:
                processing_time = random.uniform(0.1, 0.3)
                time.sleep(processing_time)
                test_scheduler.complete_request(req.request_id, f"完成: {req.prompt[:20]}...")
                processed += 1
        
        end_time = time.time()
        
        # 收集结果
        status = test_scheduler.get_status()
        results[policy.value] = {
            'total_time': end_time - start_time,
            'completed_requests': status['finished_requests'],
            'avg_wait_time': status['performance_stats']['average_wait_time'],
            'throughput': status['performance_stats']['throughput']
        }
        
        print(f"  完成请求: {status['finished_requests']}")
        print(f"  平均等待时间: {status['performance_stats']['average_wait_time']:.2f}秒")
        print(f"  吞吐量: {status['performance_stats']['throughput']:.2f} 请求/秒")
    
    # 可视化比较结果
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    policies_names = list(results.keys())
    
    # 平均等待时间
    wait_times = [results[p]['avg_wait_time'] for p in policies_names]
    axes[0].bar(policies_names, wait_times, color='skyblue')
    axes[0].set_title('⏱️ 平均等待时间')
    axes[0].set_ylabel('秒')
    axes[0].tick_params(axis='x', rotation=45)
    
    # 吞吐量
    throughputs = [results[p]['throughput'] for p in policies_names]
    axes[1].bar(policies_names, throughputs, color='lightgreen')
    axes[1].set_title('⚡ 吞吐量')
    axes[1].set_ylabel('请求/秒')
    axes[1].tick_params(axis='x', rotation=45)
    
    # 完成请求数
    completed = [results[p]['completed_requests'] for p in policies_names]
    axes[2].bar(policies_names, completed, color='lightcoral')
    axes[2].set_title('✅ 完成请求数')
    axes[2].set_ylabel('请求数')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return results


# 运行性能比较
print("📊 开始性能可视化演示...")

# 重新创建引擎进行演示
demo_memory = MemoryManager(total_memory=512)
demo_scheduler = IntelligentScheduler(
    memory_manager=demo_memory,
    policy=SchedulerPolicy.MEMORY_AWARE
)
demo_engine = ContinuousBatchingEngine(demo_scheduler)

# 启动并添加一些请求
demo_engine.start()
for i in range(8):
    req = InferenceRequest(
        request_id=f"demo_req_{i}",
        prompt=f"演示请求 {i}",
        max_tokens=random.randint(20, 60),
        priority=random.randint(1, 4)
    )
    demo_engine.add_request(req)
    time.sleep(0.2)

# 运行一段时间收集数据
time.sleep(2)
demo_engine.stop()

# 可视化性能
visualize_scheduler_performance(demo_engine)

# 比较调度策略
comparison_results = compare_scheduling_policies()

## 🚀 高级优化技术

探索更高级的优化技术，包括预测性调度和自适应批处理。

In [ ]:
class PredictiveScheduler(IntelligentScheduler):
    """预测性调度器 - 基于历史数据预测请求模式"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.request_history = []
        self.prediction_window = 60  # 预测窗口(秒)
    
    def add_request(self, request: InferenceRequest):
        """添加请求并记录历史"""
        super().add_request(request)
        
        # 记录请求模式
        self.request_history.append({
            'timestamp': request.created_at,
            'estimated_tokens': request.estimated_tokens,
            'estimated_memory': request.estimated_memory,
            'priority': request.priority
        })
        
        # 保持历史记录在合理范围内
        if len(self.request_history) > 1000:
            self.request_history = self.request_history[-500:]
    
    def predict_incoming_load(self) -> Dict[str, float]:
        """预测未来的负载"""
        if len(self.request_history) < 10:
            return {'predicted_requests': 0, 'predicted_memory': 0}
        
        current_time = time.time()
        recent_requests = [
            req for req in self.request_history
            if current_time - req['timestamp'] <= self.prediction_window
        ]
        
        if not recent_requests:
            return {'predicted_requests': 0, 'predicted_memory': 0}
        
        # 简单的线性预测
        request_rate = len(recent_requests) / self.prediction_window
        avg_memory = np.mean([req['estimated_memory'] for req in recent_requests])
        
        # 预测下一分钟的负载
        predicted_requests = request_rate * 60
        predicted_memory = predicted_requests * avg_memory
        
        return {
            'predicted_requests': predicted_requests,
            'predicted_memory': predicted_memory,
            'current_rate': request_rate
        }
    
    def adaptive_batch_size(self) -> int:
        """自适应批处理大小"""
        prediction = self.predict_incoming_load()
        memory_stats = self.memory_manager.get_memory_stats()
        
        # 基于预测负载和当前内存使用调整批处理大小
        base_batch_size = self.max_batch_size
        
        # 如果预测负载高，增加批处理大小
        if prediction['current_rate'] > 2.0:  # 每秒超过2个请求
            base_batch_size = min(base_batch_size + 2, 12)
        
        # 如果内存使用率高，减少批处理大小
        if memory_stats['utilization'] > 0.8:
            base_batch_size = max(base_batch_size - 1, 2)
        
        return base_batch_size


class AdaptiveMemoryManager(MemoryManager):
    """自适应内存管理器"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.allocation_patterns = []
        self.defragmentation_threshold = 0.7  # 碎片率阈值
    
    def allocate(self, request: InferenceRequest) -> bool:
        """智能分配内存"""
        # 检查是否需要碎片整理
        if self._calculate_fragmentation() > self.defragmentation_threshold:
            self._defragment_memory()
        
        # 记录分配模式
        allocation_success = super().allocate(request)
        
        self.allocation_patterns.append({
            'timestamp': time.time(),
            'request_memory': request.estimated_memory,
            'success': allocation_success,
            'fragmentation': self._calculate_fragmentation()
        })
        
        return allocation_success
    
    def _defragment_memory(self):
        """内存碎片整理"""
        print("🔧 执行内存碎片整理...")
        
        # 简化的碎片整理：重新组织内存块
        allocated_blocks = [(i, block) for i, block in enumerate(self.blocks) if not block.is_free]
        
        # 重置所有块
        for block in self.blocks:
            if block.is_free:
                continue
            # 保存分配信息
            temp_info = (block.request_id, block.allocated_at)
            block.is_free = True
            block.request_id = None
            block.allocated_at = None
        
        # 重新分配到连续位置
        current_pos = 0
        for original_idx, block_info in allocated_blocks:
            if current_pos < len(self.blocks):
                self.blocks[current_pos].is_free = False
                self.blocks[current_pos].request_id = block_info.request_id
                self.blocks[current_pos].allocated_at = block_info.allocated_at
                current_pos += 1
    
    def get_allocation_efficiency(self) -> float:
        """计算分配效率"""
        if not self.allocation_patterns:
            return 1.0
        
        recent_patterns = self.allocation_patterns[-100:]  # 最近100次分配
        success_rate = sum(1 for p in recent_patterns if p['success']) / len(recent_patterns)
        
        return success_rate


# 演示高级优化技术
def demonstrate_advanced_optimization():
    """演示高级优化技术"""
    print("🚀 高级优化技术演示:")
    
    # 创建自适应系统
    adaptive_memory = AdaptiveMemoryManager(total_memory=256)
    predictive_scheduler = PredictiveScheduler(
        memory_manager=adaptive_memory,
        policy=SchedulerPolicy.MEMORY_AWARE
    )
    
    # 模拟不同负载模式
    load_patterns = [
        {'name': '低负载', 'requests_per_sec': 0.5, 'duration': 10},
        {'name': '中等负载', 'requests_per_sec': 2.0, 'duration': 15},
        {'name': '高负载', 'requests_per_sec': 5.0, 'duration': 10}
    ]
    
    results = []
    
    for pattern in load_patterns:
        print(f"\n测试负载模式: {pattern['name']}")
        
        start_time = time.time()
        request_count = 0
        
        while time.time() - start_time < pattern['duration']:
            # 根据负载模式生成请求
            if random.random() < pattern['requests_per_sec'] * 0.1:  # 每0.1秒的概率
                request = InferenceRequest(
                    request_id=f"{pattern['name']}_req_{request_count}",
                    prompt=f"负载测试请求 {request_count}",
                    max_tokens=random.randint(30, 100),
                    priority=random.randint(1, 5)
                )
                predictive_scheduler.add_request(request)
                request_count += 1
            
            # 处理请求
            adaptive_batch_size = predictive_scheduler.adaptive_batch_size()
            predictive_scheduler.max_batch_size = adaptive_batch_size
            
            batch = predictive_scheduler.schedule_batch()
            for req in batch:
                # 模拟快速处理
                predictive_scheduler.complete_request(req.request_id, f"处理完成: {req.prompt[:20]}...")
            
            time.sleep(0.1)
        
        # 收集结果
        prediction = predictive_scheduler.predict_incoming_load()
        memory_efficiency = adaptive_memory.get_allocation_efficiency()
        
        pattern_result = {
            'pattern': pattern['name'],
            'requests_generated': request_count,
            'predicted_rate': prediction['current_rate'],
            'memory_efficiency': memory_efficiency,
            'final_batch_size': adaptive_batch_size
        }
        
        results.append(pattern_result)
        
        print(f"  生成请求: {request_count}")
        print(f"  预测请求率: {prediction['current_rate']:.2f} 请求/秒")
        print(f"  内存分配效率: {memory_efficiency:.2%}")
        print(f"  自适应批处理大小: {adaptive_batch_size}")
    
    return results


# 运行高级优化演示
optimization_results = demonstrate_advanced_optimization()

# 可视化优化结果
if optimization_results:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('🚀 高级优化技术效果', fontsize=14, fontweight='bold')
    
    patterns = [r['pattern'] for r in optimization_results]
    
    # 请求生成数量
    requests_generated = [r['requests_generated'] for r in optimization_results]
    axes[0, 0].bar(patterns, requests_generated, color='lightblue')
    axes[0, 0].set_title('📊 生成请求数量')
    axes[0, 0].set_ylabel('请求数')
    
    # 预测准确性
    predicted_rates = [r['predicted_rate'] for r in optimization_results]
    axes[0, 1].bar(patterns, predicted_rates, color='lightgreen')
    axes[0, 1].set_title('🔮 预测请求率')
    axes[0, 1].set_ylabel('请求/秒')
    
    # 内存效率
    memory_efficiencies = [r['memory_efficiency'] * 100 for r in optimization_results]
    axes[1, 0].bar(patterns, memory_efficiencies, color='lightcoral')
    axes[1, 0].set_title('🧠 内存分配效率')
    axes[1, 0].set_ylabel('效率 (%)')
    axes[1, 0].set_ylim(0, 100)
    
    # 自适应批处理大小
    batch_sizes = [r['final_batch_size'] for r in optimization_results]
    axes[1, 1].bar(patterns, batch_sizes, color='gold')
    axes[1, 1].set_title('⚡ 自适应批处理大小')
    axes[1, 1].set_ylabel('批处理大小')
    
    plt.tight_layout()
    plt.show()

print("\n✅ 高级调度和内存管理演示完成！")

## 📚 学习总结

### 🎯 核心概念回顾

通过本Notebook，我们深入学习了LLM推理系统中的高级调度和内存管理技术：

#### 1. 请求管理系统
- **请求生命周期**: 从创建到完成的完整流程
- **状态管理**: 等待、运行、换出、完成等状态转换
- **资源估算**: 基于prompt长度和max_tokens的内存需求预测

#### 2. 内存管理策略
- **分块管理**: 类似PagedAttention的内存分块机制
- **碎片整理**: 动态内存碎片整理和优化
- **自适应分配**: 基于历史模式的智能内存分配

#### 3. 调度算法
- **FCFS**: 先来先服务，简单但可能不够高效
- **优先级调度**: 基于请求优先级的调度策略
- **内存感知调度**: 综合考虑优先级、内存效率和等待时间

#### 4. 连续批处理
- **动态批处理**: 实时调整批处理大小和内容
- **请求流水线**: 新请求可以随时加入处理队列
- **性能监控**: 实时跟踪系统性能指标

#### 5. 高级优化技术
- **预测性调度**: 基于历史数据预测未来负载
- **自适应批处理**: 根据系统状态动态调整批处理大小
- **智能内存管理**: 自动碎片整理和效率优化

### 🔍 关键技术要点

1. **内存效率**: 通过分块管理和碎片整理提高内存利用率
2. **调度公平性**: 平衡不同优先级请求的处理机会
3. **系统吞吐量**: 通过连续批处理最大化系统处理能力
4. **延迟优化**: 减少请求等待时间，提升用户体验
5. **资源利用**: 充分利用GPU内存和计算资源

### 🚀 实际应用价值

这些技术在生产环境中的重要性：

- **成本效益**: 提高资源利用率，降低运营成本
- **用户体验**: 减少延迟，提供更好的服务质量
- **系统稳定性**: 智能调度避免资源争用和系统过载
- **可扩展性**: 支持动态负载变化和系统扩展

### 📈 下一步学习建议

1. **深入研究**: 学习更多高级调度算法（如CFS、BFS等）
2. **实际部署**: 在真实GPU环境中测试这些技术
3. **性能调优**: 针对特定硬件和模型进行参数优化
4. **分布式扩展**: 学习多GPU和多节点的调度策略

### 🎓 练习建议

1. **修改调度策略**: 尝试实现自己的调度算法
2. **优化内存管理**: 改进碎片整理算法
3. **性能基准测试**: 在不同负载下比较各种策略
4. **可视化改进**: 添加更多性能监控图表

通过这些高级技术的学习和实践，你将能够构建更高效、更稳定的LLM推理系统！